# 05 · Corners & Cards Models
Phase 5: Secondary targets — corners prediction and yellow/red cards.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.data_loader import load_all
from src.models import CornersModel, YellowCardsModel
np.random.seed(42)
data = load_all(verbose=False)
wc_stats = data['wc_stats']
wc_feat  = pd.read_csv('../data/processed/wc2026_features.csv')


## 5.1 WC Historical Stats Analysis

In [ ]:
if wc_stats is not None:
    print("WC Historical Stats columns:")
    print(wc_stats.columns.tolist())
    print(f"\nMatches: {len(wc_stats)} | Years: {wc_stats['Year'].min()}–{wc_stats['Year'].max()}")
    
    # Card analysis
    if 'home_red_card' in wc_stats.columns and 'away_red_card' in wc_stats.columns:
        hc = wc_stats['home_red_card'].fillna('').astype(str).str.count('·')
        ac = wc_stats['away_red_card'].fillna('').astype(str).str.count('·')
        total_red = hc + ac
        print(f"\nRed cards per match: mean={total_red.mean():.3f}, "
              f"std={total_red.std():.3f}, P(0)={  (total_red==0).mean():.1%}")
        
    # Goals analysis from WC stats
    recent_wc = wc_stats[wc_stats['Year'] >= 1990]
    print(f"\nRecent WC (1990-2022) — {len(recent_wc)} matches")
    goals_h = recent_wc['home_score'].dropna()
    goals_a = recent_wc['away_score'].dropna()
    print(f"Avg goals/match: {(goals_h + goals_a).mean():.2f}")
else:
    print("WC stats not available. Using empirical priors:")
    print("  Avg corners/match: 9.8 (std ~3.0)")
    print("  Avg yellow cards/match: 3.1 (std ~1.6)")
    print("  Avg red cards/match: 0.12 (P(0)~78%)")


## 5.2 Corners Model

In [ ]:
corners_model = CornersModel()
if wc_stats is not None:
    corners_model.fit_from_wc_stats(wc_stats)

print("Corners Model: Prior-based with attacking style adjustment")
print(f"  Base: {corners_model.WC_CORNERS_MEAN} corners/match")
print(f"  Adjustment: +0.5 * (attack_style_home + attack_style_away)")
print(f"  Clip range: [6, 14]")

# Test predictions
test_cases = [
    ("France", "Brazil", 1.5, 1.5, 1.0, 1.0, 100),   # attack vs attack
    ("Germany", "Morocco", 1.8, 1.2, 1.1, 1.3, 200),  # high pressure
    ("Cabo Verde", "Uruguay", 0.9, 1.5, 1.3, 1.1, -150), # underdog
]
print("\nSample predictions:")
print(f"{'Match':<30} {'Corners'}")
print("-" * 40)
for h, a, hgf, agf, hga, aga, elo_diff in test_cases:
    c = corners_model.predict(hgf, agf, hga, aga, elo_diff)
    print(f"{h} vs {a:<18} {c}")


## 5.3 Yellow Cards Model

In [ ]:
yellows_model = YellowCardsModel()
print("Yellow Cards Model: Prior + Confederation pair + Knockout adjustment")
print(f"  Base: {yellows_model.WC_YELLOW_MEAN} yellows/match")
print("\nConfederation pair adjustments:")
for pair, adj in yellows_model.CONF_CARD_ADJUSTMENTS.items():
    if adj != 0:
        print(f"  {pair[0]} vs {pair[1]}: +{adj:.1f}")

# Test predictions
test_conf = [
    ("CONMEBOL", "CONMEBOL", False, 50),
    ("UEFA", "UEFA", False, 100),
    ("CONMEBOL", "UEFA", True, 80),
    ("CAF", "CAF", False, 30),
    ("AFC", "CONCACAF", True, 200),
]
print("\nSample yellow card predictions:")
print(f"{'Home Conf':<12} {'Away Conf':<12} {'KO':<6} {'EloΔ':<8} {'Predicted'}")
print("-" * 50)
for ch, ca, ko, ed in test_conf:
    y = yellows_model.predict(ch, ca, ko, ed)
    print(f"{ch:<12} {ca:<12} {str(ko):<6} {ed:<8} {y}")


## 5.4 Red Cards Model — Constant Zero Baseline

In [ ]:
print("Red Cards Model: CONSTANT ZERO")
print("=" * 45)
print("Rationale:")
print("  - 2022 WC (64 matches): 8 red cards total → 0.125/match")
print("  - P(0 reds) ≈ 78% across WC history")
print("  - Any prediction of non-zero reds creates MORE error than baseline")
print("  - Modal prediction = 0 is the optimal constant predictor")
print("")
print("  >>> All matches: predicted_red_cards = 0")


## 5.5 Apply Models to All WC 2026 Fixtures

In [ ]:
wc_feat['corners'] = wc_feat.apply(corners_model.predict_from_row, axis=1)
wc_feat['yellow_cards'] = wc_feat.apply(yellows_model.predict_from_row, axis=1)
wc_feat['red_cards'] = 0

output = wc_feat[['match_id','group','home_team','away_team',
                   'corners','yellow_cards','red_cards']].copy()
output.to_csv('../outputs/predictions/corners_cards_predictions.csv', index=False)
print("Saved corners_cards_predictions.csv")
print(f"\nCorners summary: min={output['corners'].min()}, max={output['corners'].max()}, "
      f"mean={output['corners'].mean():.1f}")
print(f"Yellows summary: min={output['yellow_cards'].min()}, max={output['yellow_cards'].max()}, "
      f"mean={output['yellow_cards'].mean():.1f}")
print("\nSample:")
output.head(8)


## ✅ Phase 5 Complete
- Corners predictions saved
- Yellow cards model applied
- Red cards = 0 (principled decision)